In [1]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. データ読み込み
# ============================================================
print("📂 データ読み込み...")
train_df = pd.read_csv("data/train.csv", encoding="cp932")
test_df = pd.read_csv("data/test.csv", encoding="cp932")

target_col = "含水率"
meta_cols = ["sample number", "species number", "樹種", "含水率"]
spectrum_cols = [c for c in train_df.columns if c not in meta_cols]
wavenumbers = np.array([float(c) for c in spectrum_cols])

X_train_raw = train_df[spectrum_cols].to_numpy(dtype=float)
y_train = train_df[target_col].values
species_train = train_df["species number"].values
X_test_raw = test_df[spectrum_cols].to_numpy(dtype=float)

print(f"  訓練: {X_train_raw.shape}, テスト: {X_test_raw.shape}")
print(f"  波数範囲: {wavenumbers.min():.0f} ~ {wavenumbers.max():.0f} cm⁻¹")

# ============================================================
# 2. 前処理
# ============================================================
def snv(X):
    m = X.mean(axis=1, keepdims=True)
    s = X.std(axis=1, keepdims=True)
    s[s == 0] = 1.0
    return (X - m) / s

# ============================================================
# 3. 波数帯域選択関数
# ============================================================
def select_bands(X, wavenumbers, bands):
    """指定した波数帯域のみを抽出"""
    mask = np.zeros(len(wavenumbers), dtype=bool)
    for lo, hi in bands:
        mask |= (wavenumbers >= lo) & (wavenumbers <= hi)
    return X[:, mask], wavenumbers[mask]

# 水の吸収帯域定義
WATER_BANDS = {
    "OH_comb_only": [(5050, 5350)],
    "OH_1st_only":  [(6700, 7200)],
    "OH_both":      [(5050, 5350), (6700, 7200)],
    "OH_all3":      [(5050, 5350), (6700, 7200), (8200, 8800)],
    "OH+CH":        [(4000, 4500), (5050, 5350), (5600, 6000), (6700, 7200)],
    "wide_water":   [(4500, 5500), (6500, 7500)],
    "full":         [(wavenumbers.min(), wavenumbers.max())],
}

for name, bands in WATER_BANDS.items():
    _, wn_sel = select_bands(X_train_raw, wavenumbers, bands)
    print(f"  {name:20s}: {len(wn_sel)}波数")

# ============================================================
# 4. 評価関数（LOGO全体平均 + GroupKFold + テスト予測）
# ============================================================
def evaluate_full(X_tr, y, species, model_fn):
    """LOGO全樹種のRMSEリストを返す"""
    cv = LeaveOneGroupOut()
    sp_rmses = {}
    for tr_idx, va_idx in cv.split(X_tr, y, groups=species):
        sp = species[va_idx][0]
        m = model_fn()
        if isinstance(m, PLSRegression):
            m.fit(X_tr[tr_idx], y[tr_idx])
        elif isinstance(m, LGBMRegressor):
            m.fit(X_tr[tr_idx], y[tr_idx],
                  eval_set=[(X_tr[va_idx], y[va_idx])],
                  callbacks=[early_stopping(50), log_evaluation(0)])
        else:
            m.fit(X_tr[tr_idx], y[tr_idx])
        sp_rmses[sp] = np.sqrt(mean_squared_error(
            y[va_idx], m.predict(X_tr[va_idx]).ravel()))
    
    rmses = list(sp_rmses.values())
    logo_mean = np.mean(rmses)
    logo_median = np.median(rmses)
    logo_trim2 = np.mean(sorted(rmses)[:-2])
    # 新指標: 75パーセンタイル（外れ値に引きずられにくいが無視もしない）
    logo_p75 = np.percentile(rmses, 75)
    
    return {
        "logo_mean": logo_mean,
        "logo_median": logo_median,
        "logo_trim2": logo_trim2,
        "logo_p75": logo_p75,
        "sp_rmses": sp_rmses,
    }

# ============================================================
# 5. 実験: 波数帯域選択 × 前処理 × モデル
# ============================================================
print("\n" + "=" * 70)
print("🚀 実験: 波数帯域 × 前処理 × モデル")
print("=" * 70)

# 前処理パターン
prep_configs = {
    "raw":             lambda X: X,
    "SNV":             lambda X: snv(X),
    "SNV+SG1d(w=7)":  lambda X: savgol_filter(snv(X), 7, 2, deriv=1, axis=1),
    "SNV+SG1d(w=11)": lambda X: savgol_filter(snv(X), 11, 2, deriv=1, axis=1),
    "SG1d(w=7)":      lambda X: savgol_filter(X, 7, 2, deriv=1, axis=1),
    "SG1d(w=11)":     lambda X: savgol_filter(X, 11, 2, deriv=1, axis=1),
}

# モデルパターン
model_configs = {
    "PLS(2)":  lambda: PLSRegression(n_components=2),
    "PLS(3)":  lambda: PLSRegression(n_components=3),
    "PLS(5)":  lambda: PLSRegression(n_components=5),
    "PLS(7)":  lambda: PLSRegression(n_components=7),
    "Ridge(100)":  lambda: Ridge(alpha=100),
    "Ridge(1000)": lambda: Ridge(alpha=1000),
    "LGB(nl=7)":  lambda: LGBMRegressor(n_estimators=1000, learning_rate=0.05,
                      num_leaves=7, verbosity=-1, random_state=42),
    "LGB(nl=15)": lambda: LGBMRegressor(n_estimators=1000, learning_rate=0.05,
                      num_leaves=15, verbosity=-1, random_state=42),
    "LGB(nl=31)": lambda: LGBMRegressor(n_estimators=1000, learning_rate=0.05,
                      num_leaves=31, verbosity=-1, random_state=42),
}

# 全組み合わせ
experiments = []
for band_name in WATER_BANDS:
    for prep_name in prep_configs:
        # 帯域選択した後に微分する場合、波数が少なすぎるとSGが適用できない
        bands = WATER_BANDS[band_name]
        _, wn_sel = select_bands(X_train_raw, wavenumbers, bands)
        if "SG1d" in prep_name:
            w = int(prep_name.split("w=")[1].replace(")", ""))
            if len(wn_sel) < w:
                continue
        for model_name in model_configs:
            experiments.append({
                "band": band_name, "prep": prep_name, "model": model_name,
            })

print(f"  全{len(experiments)}パターン")

# 実行
all_results = []
pbar = tqdm(total=len(experiments), desc="実験進捗",
            bar_format='{l_bar}{bar:30}{r_bar}')

for exp in experiments:
    pbar.set_postfix_str(f"{exp['band'][:10]}+{exp['prep'][:10]}+{exp['model'][:10]}")
    t0 = time.time()
    
    try:
        # 波数帯域選択
        bands = WATER_BANDS[exp["band"]]
        X_tr_band, _ = select_bands(X_train_raw, wavenumbers, bands)
        
        # 前処理
        prep_fn = prep_configs[exp["prep"]]
        X_tr_prep = prep_fn(X_tr_band)
        
        # 評価
        result = evaluate_full(X_tr_prep, y_train, species_train,
                              model_configs[exp["model"]])
        elapsed = time.time() - t0
        
        all_results.append({
            "band": exp["band"], "prep": exp["prep"], "model": exp["model"],
            "logo_mean": result["logo_mean"],
            "logo_median": result["logo_median"],
            "logo_trim2": result["logo_trim2"],
            "logo_p75": result["logo_p75"],
            "time": elapsed,
            "sp_rmses": result["sp_rmses"],
        })
        
        if result["logo_median"] < 15:
            tqdm.write(f"  ⭐ {exp['band']:15s} {exp['prep']:15s} {exp['model']:12s} "
                       f"med={result['logo_median']:5.1f} "
                       f"mean={result['logo_mean']:5.1f} "
                       f"p75={result['logo_p75']:5.1f}")
    except Exception as e:
        tqdm.write(f"  ❌ Error: {e}")
    
    pbar.update(1)

pbar.close()

# ============================================================
# 6. 結果サマリー
# ============================================================
res_df = pd.DataFrame([{k: v for k, v in r.items() if k != "sp_rmses"}
                        for r in all_results])

# 複数指標で並べ替えて比較
for metric in ["logo_mean", "logo_median", "logo_p75"]:
    sorted_df = res_df.sort_values(metric)
    print(f"\n{'='*70}")
    print(f"📋 上位15（{metric}順）")
    print(f"{'='*70}")
    print(sorted_df.head(15).to_string(index=False))

# --- 波数帯域別ベスト（各指標）---
print(f"\n{'='*70}")
print(f"📋 波数帯域別ベスト")
print(f"{'='*70}")
for band_name in WATER_BANDS:
    sub = res_df[res_df["band"] == band_name]
    if len(sub) > 0:
        b_mean = sub.sort_values("logo_mean").iloc[0]
        b_med = sub.sort_values("logo_median").iloc[0]
        print(f"  {band_name:20s} | mean_best={b_mean['logo_mean']:5.1f}({b_mean['model']:12s}+{b_mean['prep']:15s}) "
              f"| med_best={b_med['logo_median']:5.1f}({b_med['model']:12s}+{b_med['prep']:15s})")

# ============================================================
# 7. Public Score との相関確認
# ============================================================
print(f"\n{'='*70}")
print(f"📌 既知のPublic Scoreとの比較")
print(f"{'='*70}")

# 過去の提出設定に最も近い実験を探す
known_submissions = [
    {"desc": "提出1(Score=21)", "band": "full", "prep": "SNV+SG1d(w=11)", "model": "LGB(nl=31)", "public": 21.0},
]

for ks in known_submissions:
    match = res_df[(res_df["band"]==ks["band"]) & 
                   (res_df["prep"]==ks["prep"]) & 
                   (res_df["model"]==ks["model"])]
    if len(match) > 0:
        r = match.iloc[0]
        print(f"\n  {ks['desc']}: Public Score = {ks['public']}")
        print(f"    logo_mean={r['logo_mean']:.2f}, logo_median={r['logo_median']:.2f}, "
              f"logo_p75={r['logo_p75']:.2f}")
        print(f"    → どの指標がPublic Scoreに近いか確認")

# ============================================================
# 8. 上位モデルの樹種別RMSE
# ============================================================
print(f"\n{'='*70}")
print(f"🔍 各指標の上位1位の樹種別RMSE")
print(f"{'='*70}")

sp_names_map = {}
for sp_num in np.unique(species_train):
    sp_names_map[sp_num] = train_df.loc[train_df['species number']==sp_num, '樹種'].iloc[0]

for metric in ["logo_mean", "logo_median"]:
    sorted_df = res_df.sort_values(metric)
    best_row = sorted_df.iloc[0]
    best_exp = [r for r in all_results if r["band"]==best_row["band"] 
                and r["prep"]==best_row["prep"] and r["model"]==best_row["model"]][0]
    
    print(f"\n  {metric} 1位: {best_row['band']} + {best_row['prep']} + {best_row['model']}")
    print(f"  mean={best_row['logo_mean']:.2f}, median={best_row['logo_median']:.2f}, "
          f"p75={best_row['logo_p75']:.2f}")
    for sp_num, rmse in sorted(best_exp["sp_rmses"].items(), key=lambda x: x[1], reverse=True):
        marker = "🔴" if rmse > 30 else "🟡" if rmse > 20 else "🟢"
        print(f"    {marker} {sp_names_map[sp_num]:15s} RMSE={rmse:6.1f}")

print(f"""
{'='*70}
📌 次のステップ
{'='*70}
1. 既知のPublic Score(21)と各LOGO指標の値を比較
   → どの指標がPublic Scoreを最もよく近似するか確認
   
2. 波数帯域選択の効果を確認
   → 水の吸収帯だけを使うと改善するか？
   → full(全波数)より良い帯域があるか？

3. 結果を見て、次に提出するモデルを検討
   → ただし提出は慎重に（1日の枠を大事に）
""")

📂 データ読み込み...
  訓練: (1322, 1555), テスト: (550, 1555)
  波数範囲: 4000 ~ 9994 cm⁻¹
  OH_comb_only        : 78波数
  OH_1st_only         : 129波数
  OH_both             : 207波数
  OH_all3             : 363波数
  OH+CH               : 440波数
  wide_water          : 518波数
  full                : 1555波数

🚀 実験: 波数帯域 × 前処理 × モデル
  全378パターン


実験進捗:   2%|▍                             | 6/378 [00:00<00:13, 26.77it/s, OH_comb_on+raw+LGB(nl=7)] 

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[321]	valid_0's l2: 952.857
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[46]	valid_0's l2: 917.528
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[53]	valid_0's l2: 192.28
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[70]	valid_0's l2: 602.391
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[54]	valid_0's l2: 388.219
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[38]	valid_0's l2: 421.375
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[458]	valid_0's l2: 143.068
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[10]	valid_0's l2: 488.996
Training until 

実験進捗:   2%|▌                             | 7/378 [00:00<00:54,  6.75it/s, OH_comb_on+raw+LGB(nl=15)]

Early stopping, best iteration is:
[18]	valid_0's l2: 166.087
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[315]	valid_0's l2: 853.502
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[46]	valid_0's l2: 1075.98
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[174]	valid_0's l2: 216.946
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[791]	valid_0's l2: 395.603
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[47]	valid_0's l2: 552.297
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[36]	valid_0's l2: 558.36
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[296]	valid_0's l2: 98.2735
Training until validation scores don't improve for 50 rounds
Early stoppin

...

Early stopping, best iteration is:
[20]	valid_0's l2: 138.132
  ⭐ full            SG1d(w=11)      LGB(nl=7)    med= 12.9 mean= 18.1 p75= 18.0
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[179]	valid_0's l2: 239.297
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[171]	valid_0's l2: 187.308
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[25]	valid_0's l2: 432.977
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[151]	valid_0's l2: 236.914
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[62]	valid_0's l2: 141.618
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[31]	valid_0's l2: 509.795
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[90]	valid_0's l2: 29

実験進捗: 100%|█████████████████████████████▉| 377/378 [19:00<00:12, 12.74s/it, full+SG1d(w=11)+LGB(nl=31)]

Early stopping, best iteration is:
[16]	valid_0's l2: 102.367
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[230]	valid_0's l2: 259.481
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[156]	valid_0's l2: 185.793
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[34]	valid_0's l2: 441.914
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[157]	valid_0's l2: 260.868
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[40]	valid_0's l2: 153.109
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[30]	valid_0's l2: 536.803
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[91]	valid_0's l2: 30.6222
Training until validation scores don't improve for 50 rounds
Did not meet 

実験進捗: 100%|██████████████████████████████| 378/378 [19:55<00:00,  3.16s/it, full+SG1d(w=11)+LGB(nl=31)]    

Early stopping, best iteration is:
[16]	valid_0's l2: 118.569
  ⭐ full            SG1d(w=11)      LGB(nl=31)   med= 13.6 mean= 19.7 p75= 21.0

📋 上位15（logo_mean順）
       band           prep      model  logo_mean  logo_median  logo_trim2  logo_p75      time
 wide_water  SNV+SG1d(w=7) LGB(nl=15)  15.008904    12.194159   11.042253 15.044321 12.907666
 wide_water  SNV+SG1d(w=7)  LGB(nl=7)  15.196194    12.198950   11.149289 15.554223  3.490758
 wide_water            SNV  LGB(nl=7)  15.379493     8.671918   10.178074 15.347593  4.940623
OH_1st_only            SNV  LGB(nl=7)  15.491258    14.243712   12.628719 16.024995  1.311785
 wide_water  SNV+SG1d(w=7) LGB(nl=31)  15.553096    12.953936   11.664133 15.201670 18.007285
    OH_all3  SNV+SG1d(w=7)  LGB(nl=7)  15.709753    11.975012   12.218312 16.595872  2.397595
    OH_both            SNV LGB(nl=15)  15.902234    10.892435   12.068228 21.392176  3.055035
 wide_water SNV+SG1d(w=11)  LGB(nl=7)  16.064275    13.088411   12.353569 17.182613  3

In [2]:
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. データ読み込み
# ============================================================
print("📂 データ読み込み...")
train_df = pd.read_csv("data/train.csv", encoding="cp932")
test_df = pd.read_csv("data/test.csv", encoding="cp932")

target_col = "含水率"
meta_cols = ["sample number", "species number", "樹種", "含水率"]
spectrum_cols = [c for c in train_df.columns if c not in meta_cols]
wavenumbers = np.array([float(c) for c in spectrum_cols])
wavelengths = 10000000 / wavenumbers  # nm に変換

print(f"  訓練: {train_df.shape}")
print(f"  テスト: {test_df.shape}")

# ============================================================
# 2. チームメンバーの工夫を合法的に再現
# ============================================================

# --- 工夫①: ベイスギ除外 ---
print("\n🔧 工夫①: ベイスギを訓練データから除外")
train_clean = train_df[train_df['樹種'] != 'ベイスギ'].reset_index(drop=True)
print(f"  除外前: {len(train_df)} → 除外後: {len(train_clean)} "
      f"(ベイスギ {len(train_df)-len(train_clean)}サンプル除外)")

X_train_raw = train_clean[spectrum_cols].to_numpy(dtype=float)
y_train_raw = train_clean[target_col].values
species_train = train_clean["species number"].values
X_test_raw = test_df[spectrum_cols].to_numpy(dtype=float)

# --- 工夫②: log1p変換 ---
print("🔧 工夫②: 含水率をlog1p変換")
y_train_log = np.log1p(y_train_raw)
print(f"  変換前: [{y_train_raw.min():.1f} ~ {y_train_raw.max():.1f}]")
print(f"  変換後: [{y_train_log.min():.2f} ~ {y_train_log.max():.2f}]")

# --- 工夫③: ROI選択 ---
print("🔧 工夫③: 重要帯域(ROI)の選択")
roi_mask = ((wavelengths >= 1350) & (wavelengths <= 1650)) | \
           ((wavelengths >= 1900) & (wavelengths <= 2300))
roi_cols_idx = np.where(roi_mask)[0]
roi_wn = wavenumbers[roi_mask]
print(f"  ROI: 1350-1650nm + 1900-2300nm")
print(f"  波数換算: {roi_wn.min():.0f}-{roi_wn.max():.0f} cm⁻¹")
print(f"  選択波数: {len(roi_cols_idx)} / {len(wavenumbers)}")

# --- 工夫④: 複数前処理の結合（合法版: ラグ・移動平均なし）---
print("🔧 工夫④: 複数前処理の結合（ラグ・移動平均なし）")

def preprocess_legal(X_raw, roi_idx):
    """合法的な前処理（各サンプル独立に処理可能）"""
    # SNV（各サンプル独立）
    m = X_raw.mean(axis=1, keepdims=True)
    s = X_raw.std(axis=1, keepdims=True)
    X_snv = (X_raw - m) / (s + 1e-8)
    
    # 1次微分（全波数、各サンプル独立）
    X_diff1 = savgol_filter(X_snv, window_length=15, polyorder=2, deriv=1, axis=1)
    
    # 2次微分（ROI限定、各サンプル独立）
    X_diff2_roi = savgol_filter(X_snv[:, roi_idx], window_length=15, polyorder=2, deriv=2, axis=1)
    
    # 結合
    X_combined = np.hstack([X_snv, X_diff1, X_diff2_roi])
    return X_combined

X_train = preprocess_legal(X_train_raw, roi_cols_idx)
X_test = preprocess_legal(X_test_raw, roi_cols_idx)

print(f"  特徴量数: {X_train.shape[1]}")
print(f"    SNV: {X_train_raw.shape[1]}")
print(f"    1次微分: {X_train_raw.shape[1]}")
print(f"    2次微分(ROI): {len(roi_cols_idx)}")

# ============================================================
# 3. LOGO評価（ベイスギ除外済み → 12種でLOGO）
# ============================================================
print("\n" + "=" * 70)
print("🔍 LOGO評価（ベイスギ除外済み、12種）")
print("=" * 70)

sp_names_map = {}
for sp_num in np.unique(species_train):
    sp_names_map[sp_num] = train_clean.loc[
        train_clean['species number']==sp_num, '樹種'].iloc[0]

def eval_logo(X_tr, y_log, species, model_fn):
    cv = LeaveOneGroupOut()
    sp_rmses = {}
    for tr_idx, va_idx in cv.split(X_tr, y_log, groups=species):
        sp = species[va_idx][0]
        m = model_fn()
        m.fit(X_tr[tr_idx], y_log[tr_idx],
              eval_set=[(X_tr[va_idx], y_log[va_idx])],
              callbacks=[lgb.early_stopping(30, verbose=False)])
        # log空間で予測 → 元に戻してRMSE計算
        preds = np.expm1(m.predict(X_tr[va_idx]))
        actual = np.expm1(y_log[va_idx])
        sp_rmses[sp] = np.sqrt(mean_squared_error(actual, preds))
    return sp_rmses

# まずデフォルト設定でLOGO確認
def make_lgb_default():
    return lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.05, num_leaves=31,
        colsample_bytree=0.2,  # 工夫⑤
        subsample=0.8, subsample_freq=1,
        min_child_samples=20,
        verbosity=-1, random_state=42, n_jobs=-1,
    )

print("\n  デフォルトLGB (colsample=0.2) でLOGO...")
t0 = time.time()
sp_rmses_default = eval_logo(X_train, y_train_log, species_train, make_lgb_default)
elapsed = time.time() - t0

rmses = list(sp_rmses_default.values())
print(f"  LOGO mean: {np.mean(rmses):.2f}, median: {np.median(rmses):.2f} ({elapsed:.1f}s)")
for sp_num, rmse in sorted(sp_rmses_default.items(), key=lambda x: x[1], reverse=True):
    marker = "🔴" if rmse > 30 else "🟡" if rmse > 20 else "🟢"
    print(f"    {marker} {sp_names_map[sp_num]:15s} RMSE={rmse:6.1f}")

# ============================================================
# 4. 各工夫の効果を分離して測定
# ============================================================
print("\n" + "=" * 70)
print("🔬 各工夫の効果測定（LOGO mean）")
print("=" * 70)

# ベースライン: ベイスギ込み + log変換なし + 全波数 + SNV+SG1d(w=11)
train_all = train_df.copy()
X_all_raw = train_all[spectrum_cols].to_numpy(dtype=float)
y_all = train_all[target_col].values
species_all = train_all["species number"].values

# ベイスギ込みのLOGO
X_all_snv_sg = savgol_filter(
    (X_all_raw - X_all_raw.mean(axis=1, keepdims=True)) / 
    (X_all_raw.std(axis=1, keepdims=True) + 1e-8),
    11, 2, deriv=1, axis=1)

def eval_logo_nolog(X_tr, y_raw, species, model_fn):
    """log変換なしのLOGO"""
    cv = LeaveOneGroupOut()
    sp_rmses = {}
    for tr_idx, va_idx in cv.split(X_tr, y_raw, groups=species):
        sp = species[va_idx][0]
        m = model_fn()
        m.fit(X_tr[tr_idx], y_raw[tr_idx],
              eval_set=[(X_tr[va_idx], y_raw[va_idx])],
              callbacks=[lgb.early_stopping(30, verbose=False)])
        preds = m.predict(X_tr[va_idx])
        sp_rmses[sp] = np.sqrt(mean_squared_error(y_raw[va_idx], preds))
    return sp_rmses

def make_lgb_simple():
    return lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.05, num_leaves=31,
        verbosity=-1, random_state=42, n_jobs=-1,
    )

ablation_results = []

# (A) ベースライン: ベイスギ込み + log変換なし + 全波数 + SNV+SG1d(w=11)
print("\n  (A) ベースライン: ベイスギ込み + logなし + 全波数 ...")
sp_a = eval_logo_nolog(X_all_snv_sg, y_all, species_all, make_lgb_simple)
rmses_a = list(sp_a.values())
print(f"      LOGO mean={np.mean(rmses_a):.2f}")
ablation_results.append(("A: ベースライン", np.mean(rmses_a)))

# (B) +ベイスギ除外
print("  (B) +ベイスギ除外 ...")
X_clean_snv_sg = savgol_filter(
    (X_train_raw - X_train_raw.mean(axis=1, keepdims=True)) / 
    (X_train_raw.std(axis=1, keepdims=True) + 1e-8),
    11, 2, deriv=1, axis=1)
sp_b = eval_logo_nolog(X_clean_snv_sg, y_train_raw, species_train, make_lgb_simple)
rmses_b = list(sp_b.values())
print(f"      LOGO mean={np.mean(rmses_b):.2f}")
ablation_results.append(("B: +ベイスギ除外", np.mean(rmses_b)))

# (C) +log1p変換
print("  (C) +log1p変換 ...")
sp_c = eval_logo(X_clean_snv_sg, y_train_log, species_train, make_lgb_simple)
rmses_c = list(sp_c.values())
print(f"      LOGO mean={np.mean(rmses_c):.2f}")
ablation_results.append(("C: +log1p変換", np.mean(rmses_c)))

# (D) +複数前処理結合
print("  (D) +複数前処理結合 ...")
sp_d = eval_logo(X_train, y_train_log, species_train, make_lgb_simple)
rmses_d = list(sp_d.values())
print(f"      LOGO mean={np.mean(rmses_d):.2f}")
ablation_results.append(("D: +複数前処理結合", np.mean(rmses_d)))

# (E) +colsample_bytree=0.2
print("  (E) +colsample_bytree=0.2 ...")
sp_e = eval_logo(X_train, y_train_log, species_train, make_lgb_default)
rmses_e = list(sp_e.values())
print(f"      LOGO mean={np.mean(rmses_e):.2f}")
ablation_results.append(("E: +colsample=0.2", np.mean(rmses_e)))

print("\n  --- Ablation結果 ---")
for name, rmse in ablation_results:
    print(f"    {name:25s}  LOGO mean = {rmse:.2f}")

# ============================================================
# 5. Optuna最適化（合法版）
# ============================================================
print("\n" + "=" * 70)
print("🚀 Optuna最適化（合法版、GroupKFold 3分割）")
print("=" * 70)

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    param = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'random_state': 42,
        'n_jobs': -1,
        'n_estimators': 500,
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'num_leaves': trial.suggest_int('num_leaves', 8, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.05, 0.3),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 40),
    }
    
    gkf = GroupKFold(n_splits=3)
    rmses = []
    for tr_idx, va_idx in gkf.split(X_train, y_train_log, groups=species_train):
        model = lgb.LGBMRegressor(**param)
        model.fit(
            X_train[tr_idx], y_train_log[tr_idx],
            eval_set=[(X_train[va_idx], y_train_log[va_idx])],
            callbacks=[lgb.early_stopping(15, verbose=False)]
        )
        preds = np.expm1(model.predict(X_train[va_idx]))
        actual = np.expm1(y_train_log[va_idx])
        rmses.append(np.sqrt(mean_squared_error(actual, preds)))
    return np.mean(rmses)

pbar = tqdm(total=30, desc="🔍 Optuna", bar_format='{l_bar}{bar:30}{r_bar}')
best_so_far = float('inf')

def optuna_callback(study, trial):
    global best_so_far
    pbar.update(1)
    if trial.value < best_so_far:
        best_so_far = trial.value
        pbar.set_postfix_str(f"Best={best_so_far:.3f} (trial {trial.number})")

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30, callbacks=[optuna_callback])
pbar.close()

print(f"\n🏆 ベストCV RMSE: {study.best_value:.4f}")
print("  パラメータ:")
for k, v in study.best_params.items():
    print(f"    {k}: {v:.6f}" if isinstance(v, float) else f"    {k}: {v}")

# ============================================================
# 6. LOGO再評価（Optunaベストパラメータ）
# ============================================================
print("\n" + "=" * 70)
print("🔍 Optunaベストパラメータで LOGO評価")
print("=" * 70)

best_params = study.best_params.copy()
best_params.update({
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'random_state': 42,
    'n_jobs': -1,
    'n_estimators': 1000,
})

def make_lgb_best():
    return lgb.LGBMRegressor(**best_params)

sp_rmses_best = eval_logo(X_train, y_train_log, species_train, make_lgb_best)
rmses_best = list(sp_rmses_best.values())
print(f"  LOGO mean: {np.mean(rmses_best):.2f}")
for sp_num, rmse in sorted(sp_rmses_best.items(), key=lambda x: x[1], reverse=True):
    marker = "🔴" if rmse > 30 else "🟡" if rmse > 20 else "🟢"
    print(f"    {marker} {sp_names_map[sp_num]:15s} RMSE={rmse:6.1f}")

# ============================================================
# 7. サマリー
# ============================================================
print(f"""
{'='*70}
📊 サマリー
{'='*70}

チームメンバーのスコア: 12.4（ラグ・移動平均込み → ルール違反の可能性）
チームメンバーのコードからの学び:
  ① ベイスギ除外
  ② log1p変換
  ③ ROI選択（OH結合音 + OH第一倍音）
  ④ 複数前処理結合（SNV + 1次微分 + 2次微分ROI）
  ⑤ colsample_bytree: 0.1~0.3

今回の合法版 LOGO mean: {np.mean(rmses_best):.2f}
  → ラグ・移動平均なしでどこまで迫れるか確認

推定Public Score: LOGO mean + 数ポイント
""")

📂 データ読み込み...
  訓練: (1322, 1559)
  テスト: (550, 1558)

🔧 工夫①: ベイスギを訓練データから除外
  除外前: 1322 → 除外後: 1210 (ベイスギ 112サンプル除外)
🔧 工夫②: 含水率をlog1p変換
  変換前: [0.8 ~ 216.1]
  変換後: [0.61 ~ 5.38]
🔧 工夫③: 重要帯域(ROI)の選択
  ROI: 1350-1650nm + 1900-2300nm
  波数換算: 4351-7406 cm⁻¹
  選択波数: 586 / 1555
🔧 工夫④: 複数前処理の結合（ラグ・移動平均なし）
  特徴量数: 3696
    SNV: 1555
    1次微分: 1555
    2次微分(ROI): 586

🔍 LOGO評価（ベイスギ除外済み、12種）

  デフォルトLGB (colsample=0.2) でLOGO...
  LOGO mean: 15.63, median: 16.79 (21.8s)
    🔴 ウォールナット         RMSE=  30.1
    🟡 ナラ              RMSE=  23.1
    🟡 チェリー            RMSE=  21.0
    🟢 米ヒバ             RMSE=  18.6
    🟢 クリ              RMSE=  17.8
    🟢 イチョウ            RMSE=  17.2
    🟢 ホワイトオーク         RMSE=  16.3
    🟢 ウエンジ            RMSE=  13.9
    🟢 スプルース           RMSE=   8.5
    🟢 ヒノキ             RMSE=   8.0
    🟢 ベイマツ            RMSE=   7.4
    🟢 トチ              RMSE=   5.6

🔬 各工夫の効果測定（LOGO mean）

  (A) ベースライン: ベイスギ込み + logなし + 全波数 ...
      LOGO mean=17.69
  (B) +ベイスギ除外 ...
      LOGO mean=14.03
  (C)

🔍 Optuna: 100%|██████████████████████████████| 30/30 [00:52<00:00,  1.74s/it, Best=16.724 (trial 28)]



🏆 ベストCV RMSE: 16.7241
  パラメータ:
    learning_rate: 0.025663
    max_depth: 7
    num_leaves: 39
    subsample: 0.609443
    colsample_bytree: 0.050903
    min_child_samples: 19

🔍 Optunaベストパラメータで LOGO評価
  LOGO mean: 15.05
    🟡 ウォールナット         RMSE=  24.2
    🟡 チェリー            RMSE=  22.2
    🟡 ナラ              RMSE=  21.4
    🟡 クリ              RMSE=  20.5
    🟢 米ヒバ             RMSE=  17.1
    🟢 ホワイトオーク         RMSE=  15.6
    🟢 イチョウ            RMSE=  15.1
    🟢 ウエンジ            RMSE=  14.0
    🟢 ヒノキ             RMSE=   8.8
    🟢 スプルース           RMSE=   8.1
    🟢 ベイマツ            RMSE=   7.2
    🟢 トチ              RMSE=   6.3

📊 サマリー

チームメンバーのスコア: 12.4（ラグ・移動平均込み → ルール違反の可能性）
チームメンバーのコードからの学び:
  ① ベイスギ除外
  ② log1p変換
  ③ ROI選択（OH結合音 + OH第一倍音）
  ④ 複数前処理結合（SNV + 1次微分 + 2次微分ROI）
  ⑤ colsample_bytree: 0.1~0.3

今回の合法版 LOGO mean: 15.05
  → ラグ・移動平均なしでどこまで迫れるか確認

推定Public Score: LOGO mean + 数ポイント



In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.5.1+cu121
True


In [ ]:
import pandas as pd
import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter
from tqdm import tqdm
import copy
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 0. PyTorch / Device
# ============================================================
if not torch.cuda.is_available():
    print("⚠️ CUDAが見つかりません。CPUで実行します。GPUを使うにはCUDA対応PyTorchが必要です。")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🧠 Device: {device}")

# ============================================================
# 1. データ読み込み（cell 2に準拠）
# ============================================================
print("📂 データ読み込み...")
train_df = pd.read_csv("data/train.csv", encoding="cp932")
test_df = pd.read_csv("data/test.csv", encoding="cp932")

target_col = "含水率"
meta_cols = ["sample number", "species number", "樹種", "含水率"]
spectrum_cols = [c for c in train_df.columns if c not in meta_cols]
wavenumbers = np.array([float(c) for c in spectrum_cols])
wavelengths = 10000000 / wavenumbers

# --- 工夫①: ベイスギ除外 ---
print("\n🔧 工夫①: ベイスギを訓練データから除外")
train_clean = train_df[train_df["樹種"] != "ベイスギ"].reset_index(drop=True)
print(f"  除外前: {len(train_df)} → 除外後: {len(train_clean)}")

X_train_raw = train_clean[spectrum_cols].to_numpy(dtype=np.float32)
y_train = train_clean[target_col].to_numpy(dtype=np.float32)
species_train = train_clean["species number"].to_numpy()
X_test_raw = test_df[spectrum_cols].to_numpy(dtype=np.float32)

# --- 工夫③: ROI選択 ---
print("🔧 工夫③: 重要帯域(ROI)の選択")
roi_mask = ((wavelengths >= 1350) & (wavelengths <= 1650)) | \
           ((wavelengths >= 1900) & (wavelengths <= 2300))
roi_cols_idx = np.where(roi_mask)[0]
print(f"  選択波数: {len(roi_cols_idx)} / {len(wavenumbers)}")

# --- 工夫④: 複数前処理の結合（合法版） ---
def preprocess_legal(X_raw, roi_idx):
    m = X_raw.mean(axis=1, keepdims=True)
    s = X_raw.std(axis=1, keepdims=True)
    X_snv = (X_raw - m) / (s + 1e-8)

    X_diff1 = savgol_filter(X_snv, window_length=15, polyorder=2, deriv=1, axis=1)
    X_diff2_roi = savgol_filter(X_snv[:, roi_idx], window_length=15, polyorder=2, deriv=2, axis=1)

    X_combined = np.hstack([X_snv, X_diff1, X_diff2_roi]).astype(np.float32)
    return X_combined

X_train_feat = preprocess_legal(X_train_raw, roi_cols_idx)
X_test_feat = preprocess_legal(X_test_raw, roi_cols_idx)
print(f"  特徴量数: {X_train_feat.shape[1]}")

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_feat).astype(np.float32)
X_test = scaler.transform(X_test_feat).astype(np.float32)


# ============================================================
# 2. NN定義（2出力: a, b）
#    pred = (a / b) * 100
#    b / 4.8 は [0.32, 0.79] を常に満たす
# ============================================================
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class RatioNet(nn.Module):
    def __init__(self, in_dim, hidden_dims=(256, 128, 64), dropout=0.15):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev, h),
                nn.ReLU(),
                nn.BatchNorm1d(h),
                nn.Dropout(dropout),
            ])
            prev = h
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(prev, 2)  # raw_a, raw_b

    def forward(self, x):
        h = self.backbone(x)
        raw = self.head(h)
        raw_a = raw[:, 0:1]
        raw_b = raw[:, 1:2]

        a = F.softplus(raw_a) + 1e-6
        b_ratio = 0.32 + 0.47 * torch.sigmoid(raw_b)
        b = 4.8 * b_ratio

        y = (a / (b + 1e-6)) * 100.0
        return y.squeeze(1), a.squeeze(1), b.squeeze(1), b_ratio.squeeze(1)


def train_one_fold(X_tr, y_tr, X_va, y_va, params, seed=42):
    set_seed(seed)

    model = RatioNet(
        in_dim=X_tr.shape[1],
        hidden_dims=params["hidden_dims"],
        dropout=params["dropout"],
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=params["weight_decay"],
    )

    tr_ds = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.float32),
    )
    tr_loader = DataLoader(tr_ds, batch_size=params["batch_size"], shuffle=True)

    va_ds = TensorDataset(
        torch.tensor(X_va, dtype=torch.float32),
        torch.tensor(y_va, dtype=torch.float32),
    )
    va_loader = DataLoader(va_ds, batch_size=params["batch_size"], shuffle=False)

    best_rmse = np.inf
    best_state = None
    wait = 0

    for _ in range(params["max_epochs"]):
        model.train()
        for xb, yb in tr_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            pred, _, _, _ = model(xb)
            loss = F.mse_loss(pred, yb)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            va_preds = []
            va_targets = []
            for xb_va, yb_va in va_loader:
                xb_va = xb_va.to(device)
                yb_va = yb_va.to(device)
                pred_va, _, _, _ = model(xb_va)
                va_preds.append(pred_va)
                va_targets.append(yb_va)
            va_pred_all = torch.cat(va_preds, dim=0)
            va_tgt_all = torch.cat(va_targets, dim=0)
            rmse = torch.sqrt(F.mse_loss(va_pred_all, va_tgt_all)).item()

        if rmse < best_rmse:
            best_rmse = rmse
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= params["patience"]:
                break

    model.load_state_dict(best_state)
    return model, best_rmse


def predict_ab(model, X_np):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X_np, dtype=torch.float32, device=device)
        pred, a, b, b_ratio = model(X_t)
    return (
        pred.detach().cpu().numpy(),
        a.detach().cpu().numpy(),
        b.detach().cpu().numpy(),
        b_ratio.detach().cpu().numpy(),
    )


# ============================================================
# 3. Optunaでハイパラ探索（GroupKFold）
# ============================================================
print("\n" + "=" * 70)
print("🚀 Optuna: PyTorch NN ハイパラ探索")
print("=" * 70)

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    n_layers = trial.suggest_int("n_layers", 2, 4)
    hidden_dims = []
    for i in range(n_layers):
        hidden_dims.append(trial.suggest_int(f"h{i}", 64, 512, step=32))

    params = {
        "hidden_dims": tuple(hidden_dims),
        "dropout": trial.suggest_float("dropout", 0.0, 0.35),
        "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-7, 5e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
        "max_epochs": trial.suggest_int("max_epochs", 120, 260, step=20),
        "patience": 30,
    }

    gkf = GroupKFold(n_splits=3)
    rmses = []

    for fold_id, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups=species_train), start=1):
        _, fold_rmse = train_one_fold(
            X_train[tr_idx], y_train[tr_idx],
            X_train[va_idx], y_train[va_idx],
            params=params,
            seed=2026 + fold_id,
        )
        rmses.append(fold_rmse)

    return float(np.mean(rmses))


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print(f"\n🏆 Best CV RMSE: {study.best_value:.4f}")
print("  Best params:")
for k, v in study.best_params.items():
    print(f"    {k}: {v}")

# best params を学習用dictに変換
best_hidden_dims = tuple(study.best_params[f"h{i}"] for i in range(study.best_params["n_layers"]))
best_train_params = {
    "hidden_dims": best_hidden_dims,
    "dropout": study.best_params["dropout"],
    "lr": study.best_params["lr"],
    "weight_decay": study.best_params["weight_decay"],
    "batch_size": study.best_params["batch_size"],
    "max_epochs": study.best_params["max_epochs"],
    "patience": 30,
}


# ============================================================
# 4. LOGO評価（best params）
# ============================================================
print("\n" + "=" * 70)
print("🔍 LOGO評価（Optuna最良パラメータ）")
print("=" * 70)

logo = LeaveOneGroupOut()
sp_rmses = {}
sp_names_map = {
    sp: train_clean.loc[train_clean["species number"] == sp, "樹種"].iloc[0]
    for sp in np.unique(species_train)
}

for fold_id, (tr_idx, va_idx) in enumerate(tqdm(list(logo.split(X_train, y_train, groups=species_train)), desc="LOGO"), start=1):
    model, _ = train_one_fold(
        X_train[tr_idx], y_train[tr_idx],
        X_train[va_idx], y_train[va_idx],
        params=best_train_params,
        seed=4040 + fold_id,
    )
    pred_va, _, _, b_ratio_va = predict_ab(model, X_train[va_idx])

    sp = species_train[va_idx][0]
    rmse = np.sqrt(mean_squared_error(y_train[va_idx], pred_va))
    sp_rmses[sp] = rmse

rmses = list(sp_rmses.values())
print(f"\nNN LOGO mean: {np.mean(rmses):.2f}, median: {np.median(rmses):.2f}")
for sp_num, rmse in sorted(sp_rmses.items(), key=lambda x: x[1], reverse=True):
    marker = "🔴" if rmse > 30 else "🟡" if rmse > 20 else "🟢"
    print(f"  {marker} {sp_names_map[sp_num]:15s} RMSE={rmse:6.2f}")


# ============================================================
# 5. 最終学習してテスト予測
# ============================================================
print("\n" + "=" * 70)
print("🎯 最終学習 & テスト予測")
print("=" * 70)

# 早期終了のため GroupKFold の1 splitをバリデーションとして使う
gkf = GroupKFold(n_splits=3)
tr_idx, va_idx = next(gkf.split(X_train, y_train, groups=species_train))

final_model, _ = train_one_fold(
    X_train[tr_idx], y_train[tr_idx],
    X_train[va_idx], y_train[va_idx],
    params=best_train_params,
    seed=9999,
)

train_pred, train_a, train_b, train_b_ratio = predict_ab(final_model, X_train)
test_pred, test_a, test_b, test_b_ratio = predict_ab(final_model, X_test)

print(f"Train RMSE (reference): {np.sqrt(mean_squared_error(y_train, train_pred)):.3f}")
print(f"b/4.8 train range: [{train_b_ratio.min():.3f}, {train_b_ratio.max():.3f}]")
print(f"b/4.8 test  range: [{test_b_ratio.min():.3f}, {test_b_ratio.max():.3f}]")

# pred = (a/b)*100 を採用。非負制約のみ追加
test_pred = np.clip(test_pred, 0, None)

sub_df = pd.DataFrame({
    "sample number": test_df["sample number"],
    "含水率": test_pred
})
out_sub = "data/sub_nn_ratio_ab_torch_optuna.csv"
sub_df.to_csv(out_sub, index=False, header=False ,encoding="cp932")

# a,bの確認用
debug_df = pd.DataFrame({
    "sample number": test_df["sample number"],
    "a": test_a,
    "b": test_b,
    "b_div_4_8": test_b_ratio,
    "pred": test_pred,
})
out_dbg = "data/sub_nn_ratio_ab_torch_optuna_debug.csv"
debug_df.to_csv(out_dbg, index=False, header=False, encoding="cp932")

print(f"\n✅ 保存完了: {out_sub}")
print(f"✅ 保存完了: {out_dbg}")

/home/hpc/Araki/Signate_Spectrum/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🧠 Device: cuda
📂 データ読み込み...

🔧 工夫①: ベイスギを訓練データから除外
  除外前: 1322 → 除外後: 1210
🔧 工夫③: 重要帯域(ROI)の選択
  選択波数: 586 / 1555
  特徴量数: 3696

🚀 Optuna: PyTorch NN ハイパラ探索

🏆 Best CV RMSE: 10.2645
  Best params:
    n_layers: 3
    h0: 160
    h1: 288
    h2: 512
    dropout: 0.34687268567093954
    lr: 0.002475945195886692
    weight_decay: 2.8728202659500513e-07
    batch_size: 128
    max_epochs: 260

🔍 LOGO評価（Optuna最良パラメータ）


LOGO: 100%|██████████| 12/12 [00:16<00:00,  1.34s/it]



NN LOGO mean: 9.90, median: 6.99
  🟡 ウォールナット         RMSE= 27.87
  🟢 チェリー            RMSE= 15.60
  🟢 ホワイトオーク         RMSE= 15.16
  🟢 ウエンジ            RMSE= 14.22
  🟢 イチョウ            RMSE=  9.97
  🟢 クリ              RMSE=  7.02
  🟢 米ヒバ             RMSE=  6.96
  🟢 トチ              RMSE=  6.12
  🟢 ナラ              RMSE=  4.25
  🟢 ベイマツ            RMSE=  4.02
  🟢 ヒノキ             RMSE=  3.83
  🟢 スプルース           RMSE=  3.75

🎯 最終学習 & テスト予測
Train RMSE (reference): 9.120
b/4.8 train range: [0.320, 0.790]
b/4.8 test  range: [0.320, 0.790]

✅ 保存完了: data/sub_nn_ratio_ab_torch_optuna.csv
✅ 保存完了: data/sub_nn_ratio_ab_torch_optuna_debug.csv


In [ ]:
import numpy as np
import pandas as pd
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter
from tqdm import tqdm
import copy
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# Transformer版（Optuna拡張・単独実行可能）
# 方針:
# - 時系列としてではなく「各特徴量を1トークン」として扱う
# - 各特徴量トークンに対して self-attention の Q/K/V を計算
# ============================================================

# ------------------------------
# 1. データ読み込み + 前処理（Cell 4非依存）
# ------------------------------
print("📂 Transformer用データ準備...")
train_df = pd.read_csv("data/train.csv", encoding="cp932")
test_df = pd.read_csv("data/test.csv", encoding="cp932")

target_col = "含水率"
meta_cols = ["sample number", "species number", "樹種", "含水率"]
spectrum_cols = [c for c in train_df.columns if c not in meta_cols]
wavenumbers = np.array([float(c) for c in spectrum_cols])
wavelengths = 10000000 / wavenumbers

# Cell 2と同じくベイスギ除外
train_clean = train_df[train_df["樹種"] != "ベイスギ"].reset_index(drop=True)
X_train_raw = train_clean[spectrum_cols].to_numpy(dtype=np.float32)
y_train = train_clean[target_col].to_numpy(dtype=np.float32)
species_train = train_clean["species number"].to_numpy()
X_test_raw = test_df[spectrum_cols].to_numpy(dtype=np.float32)

# Cell 2と同じROI
roi_mask = ((wavelengths >= 1350) & (wavelengths <= 1650)) | \
           ((wavelengths >= 1900) & (wavelengths <= 2300))
roi_cols_idx = np.where(roi_mask)[0]


def preprocess_legal(X_raw, roi_idx):
    m = X_raw.mean(axis=1, keepdims=True)
    s = X_raw.std(axis=1, keepdims=True)
    X_snv = (X_raw - m) / (s + 1e-8)

    X_diff1 = savgol_filter(X_snv, window_length=15, polyorder=2, deriv=1, axis=1)
    X_diff2_roi = savgol_filter(X_snv[:, roi_idx], window_length=15, polyorder=2, deriv=2, axis=1)

    return np.hstack([X_snv, X_diff1, X_diff2_roi]).astype(np.float32)


X_train_feat = preprocess_legal(X_train_raw, roi_cols_idx)
X_test_feat = preprocess_legal(X_test_raw, roi_cols_idx)

scaler = StandardScaler()
X_train_all = scaler.fit_transform(X_train_feat).astype(np.float32)
X_test_all = scaler.transform(X_test_feat).astype(np.float32)

print(f"  Train shape: {X_train_all.shape}, Test shape: {X_test_all.shape}")


# ------------------------------
# 2. Device / Seed
# ------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🧠 Transformer Device: {device}")


def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def to_gib(n_bytes):
    return n_bytes / (1024 ** 3)


# ------------------------------
# 3. 特徴量トークンTransformer
# ------------------------------
class FeatureTransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, ff_mult=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * ff_mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * ff_mult, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        # 各特徴量トークンに対してQ/K/Vを計算
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + self.drop(attn_out))
        ff_out = self.ffn(x)
        x = self.norm2(x + self.drop(ff_out))
        return x


class FeatureTokenTransformerRegressor(nn.Module):
    def __init__(self, n_features, d_model=64, n_heads=4, n_layers=2, ff_mult=4, dropout=0.1):
        super().__init__()
        self.feature_embed = nn.Linear(1, d_model)
        self.feature_bias = nn.Parameter(torch.zeros(1, n_features, d_model))
        self.blocks = nn.ModuleList([
            FeatureTransformerBlock(d_model=d_model, n_heads=n_heads, ff_mult=ff_mult, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.head = nn.Linear(d_model, 2)  # raw_a, raw_b

    def forward(self, x):
        # x: [B, F]
        h = self.feature_embed(x.unsqueeze(-1)) + self.feature_bias
        for blk in self.blocks:
            h = blk(h)

        pooled = h.mean(dim=1)
        raw = self.head(pooled)
        raw_a = raw[:, 0:1]
        raw_b = raw[:, 1:2]

        a = F.softplus(raw_a) + 1e-6
        b_ratio = 0.32 + 0.47 * torch.sigmoid(raw_b)
        b = 4.8 * b_ratio
        pred = (a / (b + 1e-6)) * 100.0

        return pred.squeeze(1), a.squeeze(1), b.squeeze(1), b_ratio.squeeze(1)


def train_transformer_one_fold(X_tr, y_tr, X_va, y_va, params, seed=42):
    set_seed(seed)

    model = FeatureTokenTransformerRegressor(
        n_features=X_tr.shape[1],
        d_model=params["d_model"],
        n_heads=params["n_heads"],
        n_layers=params["n_layers"],
        ff_mult=params["ff_mult"],
        dropout=params["dropout"],
    ).to(device)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=params["weight_decay"],
    )

    tr_ds = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.float32),
    )
    tr_loader = DataLoader(tr_ds, batch_size=params["batch_size"], shuffle=True)

    va_ds = TensorDataset(
        torch.tensor(X_va, dtype=torch.float32),
        torch.tensor(y_va, dtype=torch.float32),
    )
    va_loader = DataLoader(va_ds, batch_size=params["batch_size"], shuffle=False)

    best_rmse = np.inf
    best_state = None
    wait = 0

    for _ in range(params["max_epochs"]):
        # 1エポック = 学習データ全体を DataLoader で1周
        model.train()
        for xb, yb in tr_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            pred, _, _, _ = model(xb)
            loss = F.mse_loss(pred, yb)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        with torch.no_grad():
            va_preds = []
            va_targets = []
            for xb_va, yb_va in va_loader:
                xb_va = xb_va.to(device)
                yb_va = yb_va.to(device)
                pred_va, _, _, _ = model(xb_va)
                va_preds.append(pred_va)
                va_targets.append(yb_va)
            va_pred_all = torch.cat(va_preds, dim=0)
            va_tgt_all = torch.cat(va_targets, dim=0)
            rmse = torch.sqrt(F.mse_loss(va_pred_all, va_tgt_all)).item()

        if rmse < best_rmse:
            best_rmse = rmse
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= params["patience"]:
                break

    model.load_state_dict(best_state)
    return model, best_rmse


def predict_transformer(model, X_np):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X_np, dtype=torch.float32, device=device)
        pred, a, b, b_ratio = model(X_t)
    return (
        pred.detach().cpu().numpy(),
        a.detach().cpu().numpy(),
        b.detach().cpu().numpy(),
        b_ratio.detach().cpu().numpy(),
    )


# ------------------------------
# 4. 特徴量トークン設定（全特徴量を使用）
# ------------------------------
# 要望に合わせて特徴量は削減せず、前処理後の全特徴量をそのまま使用
X_train = X_train_all
X_test = X_test_all
print(f"ℹ️ 特徴量トークン数: {X_train.shape[1]} (削減なし)")

# VRAMに収まるかの目安（データ本体）
data_bytes = X_train.nbytes + X_test.nbytes + y_train.nbytes
print(f"📦 データサイズ目安: {to_gib(data_bytes):.3f} GiB")

if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(device).total_memory
    print(f"🧮 GPU総VRAM: {to_gib(total_vram):.2f} GiB")
    print(f"📊 データ/総VRAM比: {100.0 * data_bytes / total_vram:.2f}%")


# ------------------------------
# 5. Optuna探索（Transformer専用）
# ------------------------------
print("\n" + "=" * 70)
print("🚀 Optuna: Feature-Token Transformer")
print("=" * 70)

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective_transformer(trial):
    d_model = trial.suggest_categorical("d_model", [32, 64, 96, 128])
    n_heads_candidates = [h for h in [1, 2, 4, 8] if d_model % h == 0]
    n_heads = trial.suggest_categorical("n_heads", n_heads_candidates)

    params = {
        "d_model": d_model,
        "n_heads": n_heads,
        "n_layers": trial.suggest_int("n_layers", 1, 3),
        "ff_mult": trial.suggest_categorical("ff_mult", [2, 4, 6]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.35),
        "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-7, 5e-3, log=True),
        "batch_size": 32,
        "max_epochs": trial.suggest_int("max_epochs", 80, 180, step=20),
        "patience": 20,
    }

    gkf = GroupKFold(n_splits=3)
    rmses = []

    for fold_id, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups=species_train), start=1):
        _, fold_rmse = train_transformer_one_fold(
            X_train[tr_idx], y_train[tr_idx],
            X_train[va_idx], y_train[va_idx],
            params=params,
            seed=9000 + fold_id,
        )
        rmses.append(fold_rmse)

    return float(np.mean(rmses))


N_TRIALS = 20
study_tf = optuna.create_study(direction="minimize")
optuna_pbar = tqdm(total=N_TRIALS, desc="Optuna Trials", bar_format="{l_bar}{bar:30}{r_bar}")
best_so_far = {"value": np.inf}


def optuna_callback(study, trial):
    optuna_pbar.update(1)
    if trial.value is not None and trial.value < best_so_far["value"]:
        best_so_far["value"] = trial.value
    optuna_pbar.set_postfix_str(f"Best={best_so_far['value']:.4f} (trial {trial.number})")


study_tf.optimize(objective_transformer, n_trials=N_TRIALS, callbacks=[optuna_callback])
optuna_pbar.close()

print(f"\n🏆 Best CV RMSE: {study_tf.best_value:.4f}")
print("  Best params:")
for k, v in study_tf.best_params.items():
    print(f"    {k}: {v}")

best_params_tf = {
    "d_model": study_tf.best_params["d_model"],
    "n_heads": study_tf.best_params["n_heads"],
    "n_layers": study_tf.best_params["n_layers"],
    "ff_mult": study_tf.best_params["ff_mult"],
    "dropout": study_tf.best_params["dropout"],
    "lr": study_tf.best_params["lr"],
    "weight_decay": study_tf.best_params["weight_decay"],
    "batch_size": 32,
    "max_epochs": study_tf.best_params["max_epochs"],
    "patience": 20,
}

# VRAMピーク実測（best paramsで1バッチ）
if torch.cuda.is_available():
    model_probe = FeatureTokenTransformerRegressor(
        n_features=X_train.shape[1],
        d_model=best_params_tf["d_model"],
        n_heads=best_params_tf["n_heads"],
        n_layers=best_params_tf["n_layers"],
        ff_mult=best_params_tf["ff_mult"],
        dropout=best_params_tf["dropout"],
    ).to(device)
    bs_probe = int(best_params_tf["batch_size"])
    xb_probe = torch.tensor(X_train[:bs_probe], dtype=torch.float32, device=device)
    yb_probe = torch.tensor(y_train[:bs_probe], dtype=torch.float32, device=device)

    torch.cuda.reset_peak_memory_stats(device)
    model_probe.train()
    pred_probe, _, _, _ = model_probe(xb_probe)
    loss_probe = F.mse_loss(pred_probe, yb_probe)
    loss_probe.backward()

    peak_alloc = torch.cuda.max_memory_allocated(device)
    peak_reserved = torch.cuda.max_memory_reserved(device)
    total_vram = torch.cuda.get_device_properties(device).total_memory

    print(f"📈 1バッチ学習ピーク(alloc): {to_gib(peak_alloc):.3f} GiB")
    print(f"📈 1バッチ学習ピーク(reserved): {to_gib(peak_reserved):.3f} GiB")
    print(f"📊 ピーク/総VRAM比(alloc): {100.0 * peak_alloc / total_vram:.2f}%")

    del model_probe, xb_probe, yb_probe, pred_probe, loss_probe
    torch.cuda.empty_cache()


# ------------------------------
# 6. LOGO評価（best params）
# ------------------------------
print("\n" + "=" * 70)
print("🔍 LOGO評価（Transformer best params）")
print("=" * 70)

logo = LeaveOneGroupOut()
sp_rmses = {}
sp_names_map = {
    sp: train_clean.loc[train_clean["species number"] == sp, "樹種"].iloc[0]
    for sp in np.unique(species_train)
}

for fold_id, (tr_idx, va_idx) in enumerate(tqdm(list(logo.split(X_train, y_train, groups=species_train)), desc="LOGO"), start=1):
    model_tf, _ = train_transformer_one_fold(
        X_train[tr_idx], y_train[tr_idx],
        X_train[va_idx], y_train[va_idx],
        params=best_params_tf,
        seed=10000 + fold_id,
    )
    pred_va, _, _, b_ratio_va = predict_transformer(model_tf, X_train[va_idx])

    sp = species_train[va_idx][0]
    rmse = np.sqrt(mean_squared_error(y_train[va_idx], pred_va))
    sp_rmses[sp] = rmse

rmses = list(sp_rmses.values())
print(f"\nTransformer LOGO mean: {np.mean(rmses):.2f}, median: {np.median(rmses):.2f}")
for sp_num, rmse in sorted(sp_rmses.items(), key=lambda x: x[1], reverse=True):
    marker = "🔴" if rmse > 30 else "🟡" if rmse > 20 else "🟢"
    print(f"  {marker} {sp_names_map[sp_num]:15s} RMSE={rmse:6.2f}")


# ------------------------------
# 7. 最終学習 & 予測保存
# ------------------------------
print("\n" + "=" * 70)
print("🎯 最終学習 & テスト予測")
print("=" * 70)

gkf = GroupKFold(n_splits=3)
tr_idx, va_idx = next(gkf.split(X_train, y_train, groups=species_train))

final_tf, _ = train_transformer_one_fold(
    X_train[tr_idx], y_train[tr_idx],
    X_train[va_idx], y_train[va_idx],
    params=best_params_tf,
    seed=11111,
)

train_pred_tf, train_a_tf, train_b_tf, train_b_ratio_tf = predict_transformer(final_tf, X_train)
test_pred_tf, test_a_tf, test_b_tf, test_b_ratio_tf = predict_transformer(final_tf, X_test)

print(f"Train RMSE (reference): {np.sqrt(mean_squared_error(y_train, train_pred_tf)):.3f}")
print(f"b/4.8 train range: [{train_b_ratio_tf.min():.3f}, {train_b_ratio_tf.max():.3f}]")
print(f"b/4.8 test  range: [{test_b_ratio_tf.min():.3f}, {test_b_ratio_tf.max():.3f}]")

test_pred_tf = np.clip(test_pred_tf, 0, None)

sub_tf = pd.DataFrame({
    "sample number": test_df["sample number"],
    "含水率": test_pred_tf,
})
out_sub_tf = "data/sub_transformer_feature_tokens_optuna.csv"
sub_tf.to_csv(out_sub_tf, index=False, encoding="cp932")

dbg_tf = pd.DataFrame({
    "sample number": test_df["sample number"],
    "a": test_a_tf,
    "b": test_b_tf,
    "b_div_4_8": test_b_ratio_tf,
    "pred": test_pred_tf,
})
out_dbg_tf = "data/sub_transformer_feature_tokens_optuna_debug.csv"
dbg_tf.to_csv(out_dbg_tf, index=False, encoding="cp932")

print(f"✅ 保存完了: {out_sub_tf}")
print(f"✅ 保存完了: {out_dbg_tf}")

📂 Transformer用データ準備...
  Train shape: (1210, 3696), Test shape: (550, 3696)
🧠 Transformer Device: cuda
ℹ️ 特徴量トークン数: 3696 (削減なし)
📦 データサイズ目安: 0.024 GiB
🧮 GPU総VRAM: 31.61 GiB
📊 データ/総VRAM比: 0.08%

🚀 Optuna: Feature-Token Transformer


Optuna Trials:   0%|                              | 0/20 [00:32<?, ?it/s]
[W 2026-04-08 18:56:13,951] Trial 0 failed with parameters: {'d_model': 64, 'n_heads': 2, 'n_layers': 1, 'ff_mult': 6, 'dropout': 0.19303829485962076, 'lr': 0.00017396621409132681, 'weight_decay': 1.4802803965155571e-05, 'max_epochs': 140} because of the following error: OutOfMemoryError('CUDA out of memory. Tried to allocate 42.34 GiB. GPU 0 has a total capacity of 31.61 GiB of which 28.33 GiB is free. Including non-PyTorch memory, this process has 3.26 GiB memory in use. Of the allocated memory 1.70 GiB is allocated by PyTorch, and 1.19 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)').
Traceback (most recent call last):
  File "/home/hpc/Araki/Signate_Spectrum/.venv/lib/pytho

OutOfMemoryError: CUDA out of memory. Tried to allocate 42.34 GiB. GPU 0 has a total capacity of 31.61 GiB of which 28.33 GiB is free. Including non-PyTorch memory, this process has 3.26 GiB memory in use. Of the allocated memory 1.70 GiB is allocated by PyTorch, and 1.19 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [9]:
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("🚀 AMP retry: full features + minibatch + need_weights=False")
print(f"  X_train shape: {X_train.shape}")
print(f"  GPU device: {device}")


class FeatureTransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, ff_mult=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * ff_mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * ff_mult, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x, need_weights=False)
        x = self.norm1(x + self.drop(attn_out))
        ff_out = self.ffn(x)
        x = self.norm2(x + self.drop(ff_out))
        return x


class FeatureTokenTransformerRegressor(nn.Module):
    def __init__(self, n_features, d_model=64, n_heads=4, n_layers=2, ff_mult=4, dropout=0.1):
        super().__init__()
        self.feature_embed = nn.Linear(1, d_model)
        self.feature_bias = nn.Parameter(torch.zeros(1, n_features, d_model))
        self.blocks = nn.ModuleList([
            FeatureTransformerBlock(d_model=d_model, n_heads=n_heads, ff_mult=ff_mult, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.head = nn.Linear(d_model, 2)

    def forward(self, x):
        h = self.feature_embed(x.unsqueeze(-1)) + self.feature_bias
        for blk in self.blocks:
            h = blk(h)
        pooled = h.mean(dim=1)
        raw = self.head(pooled)
        raw_a = raw[:, 0:1]
        raw_b = raw[:, 1:2]
        a = F.softplus(raw_a) + 1e-6
        b_ratio = 0.32 + 0.47 * torch.sigmoid(raw_b)
        b = 4.8 * b_ratio
        pred = (a / (b + 1e-6)) * 100.0
        return pred.squeeze(1), a.squeeze(1), b.squeeze(1), b_ratio.squeeze(1)


def train_transformer_one_fold_amp(X_tr, y_tr, X_va, y_va, params, seed=42):
    set_seed(seed)

    model = FeatureTokenTransformerRegressor(
        n_features=X_tr.shape[1],
        d_model=params["d_model"],
        n_heads=params["n_heads"],
        n_layers=params["n_layers"],
        ff_mult=params["ff_mult"],
        dropout=params["dropout"],
    ).to(device)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=params["weight_decay"],
    )

    tr_ds = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.float32),
    )
    tr_loader = DataLoader(tr_ds, batch_size=params["batch_size"], shuffle=True)

    va_ds = TensorDataset(
        torch.tensor(X_va, dtype=torch.float32),
        torch.tensor(y_va, dtype=torch.float32),
    )
    va_loader = DataLoader(va_ds, batch_size=params["batch_size"], shuffle=False)

    scaler_amp = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_rmse = np.inf
    best_state = None
    wait = 0

    for _ in range(params["max_epochs"]):
        model.train()
        for xb, yb in tr_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                pred, _, _, _ = model(xb)
                loss = F.mse_loss(pred, yb)

            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_amp.step(opt)
            scaler_amp.update()

        model.eval()
        with torch.no_grad():
            va_preds = []
            va_targets = []
            for xb_va, yb_va in va_loader:
                xb_va = xb_va.to(device)
                yb_va = yb_va.to(device)
                with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                    pred_va, _, _, _ = model(xb_va)
                va_preds.append(pred_va)
                va_targets.append(yb_va)
            va_pred_all = torch.cat(va_preds, dim=0)
            va_tgt_all = torch.cat(va_targets, dim=0)
            rmse = torch.sqrt(F.mse_loss(va_pred_all, va_tgt_all)).item()

        if rmse < best_rmse:
            best_rmse = rmse
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= params["patience"]:
                break

    model.load_state_dict(best_state)
    return model, best_rmse


def predict_transformer_amp(model, X_np):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X_np, dtype=torch.float32, device=device)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            pred, a, b, b_ratio = model(X_t)
    return (
        pred.detach().cpu().numpy(),
        a.detach().cpu().numpy(),
        b.detach().cpu().numpy(),
        b_ratio.detach().cpu().numpy(),
    )


def objective_transformer_amp(trial):
    d_model = trial.suggest_categorical("d_model", [32, 64, 96, 128])
    n_heads_candidates = [h for h in [1, 2, 4, 8] if d_model % h == 0]
    n_heads = trial.suggest_categorical("n_heads", n_heads_candidates)

    params = {
        "d_model": d_model,
        "n_heads": n_heads,
        "n_layers": trial.suggest_int("n_layers", 1, 3),
        "ff_mult": trial.suggest_categorical("ff_mult", [2, 4, 6]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.35),
        "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-7, 5e-3, log=True),
        "batch_size": 32,
        "max_epochs": trial.suggest_int("max_epochs", 80, 180, step=20),
        "patience": 20,
    }

    gkf = GroupKFold(n_splits=3)
    rmses = []
    for fold_id, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups=species_train), start=1):
        _, fold_rmse = train_transformer_one_fold_amp(
            X_train[tr_idx], y_train[tr_idx],
            X_train[va_idx], y_train[va_idx],
            params=params,
            seed=9000 + fold_id,
        )
        rmses.append(fold_rmse)

    return float(np.mean(rmses))


AMP_TRIALS = 5
study_tf_amp = optuna.create_study(direction="minimize")
amp_pbar = tqdm(total=AMP_TRIALS, desc="AMP Optuna", bar_format="{l_bar}{bar:30}{r_bar}")
amp_best = {"value": np.inf}


def amp_callback(study, trial):
    amp_pbar.update(1)
    if trial.value is not None and trial.value < amp_best["value"]:
        amp_best["value"] = trial.value
    amp_pbar.set_postfix_str(f"Best={amp_best['value']:.4f} (trial {trial.number})")


study_tf_amp.optimize(objective_transformer_amp, n_trials=AMP_TRIALS, callbacks=[amp_callback], gc_after_trial=True)
amp_pbar.close()

print(f"\n🏆 AMP best CV RMSE: {study_tf_amp.best_value:.4f}")
print("  Best params:")
for k, v in study_tf_amp.best_params.items():
    print(f"    {k}: {v}")

best_params_tf_amp = {
    "d_model": study_tf_amp.best_params["d_model"],
    "n_heads": study_tf_amp.best_params["n_heads"],
    "n_layers": study_tf_amp.best_params["n_layers"],
    "ff_mult": study_tf_amp.best_params["ff_mult"],
    "dropout": study_tf_amp.best_params["dropout"],
    "lr": study_tf_amp.best_params["lr"],
    "weight_decay": study_tf_amp.best_params["weight_decay"],
    "batch_size": 32,
    "max_epochs": study_tf_amp.best_params["max_epochs"],
    "patience": 20,
}

print("\n🔍 AMP LOGO evaluation")
logo = LeaveOneGroupOut()
sp_rmses_amp = {}
sp_names_map = {
    sp: train_clean.loc[train_clean["species number"] == sp, "樹種"].iloc[0]
    for sp in np.unique(species_train)
}

for fold_id, (tr_idx, va_idx) in enumerate(tqdm(list(logo.split(X_train, y_train, groups=species_train)), desc="AMP LOGO"), start=1):
    model_tf_amp, _ = train_transformer_one_fold_amp(
        X_train[tr_idx], y_train[tr_idx],
        X_train[va_idx], y_train[va_idx],
        params=best_params_tf_amp,
        seed=10000 + fold_id,
    )
    pred_va, _, _, _ = predict_transformer_amp(model_tf_amp, X_train[va_idx])
    sp = species_train[va_idx][0]
    sp_rmses_amp[sp] = np.sqrt(mean_squared_error(y_train[va_idx], pred_va))

rmses_amp = list(sp_rmses_amp.values())
print(f"\nAMP Transformer LOGO mean: {np.mean(rmses_amp):.2f}, median: {np.median(rmses_amp):.2f}")
for sp_num, rmse in sorted(sp_rmses_amp.items(), key=lambda x: x[1], reverse=True):
    marker = "🔴" if rmse > 30 else "🟡" if rmse > 20 else "🟢"
    print(f"  {marker} {sp_names_map[sp_num]:15s} RMSE={rmse:6.2f}")


🚀 AMP retry: full features + minibatch + need_weights=False
  X_train shape: (1210, 3696)
  GPU device: cuda


AMP Optuna: 100%|██████████████████████████████| 5/5 [10:17<00:00, 123.48s/it, Best=17.1929 (trial 4)]



🏆 AMP best CV RMSE: 17.1929
  Best params:
    d_model: 96
    n_heads: 1
    n_layers: 1
    ff_mult: 2
    dropout: 0.24402714282624666
    lr: 0.0017534359909732863
    weight_decay: 0.0004703133229763381
    max_epochs: 180

🔍 AMP LOGO evaluation


AMP LOGO: 100%|██████████| 12/12 [03:18<00:00, 16.50s/it]


AMP Transformer LOGO mean: 11.92, median: 7.95
  🔴 ウエンジ            RMSE= 37.66
  🟢 チェリー            RMSE= 18.99
  🟢 イチョウ            RMSE= 16.84
  🟢 ホワイトオーク         RMSE= 12.46
  🟢 ベイマツ            RMSE=  9.74
  🟢 ウォールナット         RMSE=  8.12
  🟢 スプルース           RMSE=  7.79
  🟢 クリ              RMSE=  7.70
  🟢 ヒノキ             RMSE=  7.37
  🟢 トチ              RMSE=  6.46
  🟢 ナラ              RMSE=  5.47
  🟢 米ヒバ             RMSE=  4.47


In [10]:
# ============================================================
# 8. AMP版の最終学習 & 予測保存
# ============================================================
print("\n" + "=" * 70)
print("🎯 AMP版 最終学習 & テスト予測")
print("=" * 70)

# 既に得られた best_params_tf_amp を使って最終学習
# early stopping 用の検証分割を1つ使う
amp_gkf = GroupKFold(n_splits=3)
amp_tr_idx, amp_va_idx = next(amp_gkf.split(X_train, y_train, groups=species_train))

final_tf_amp, _ = train_transformer_one_fold_amp(
    X_train[amp_tr_idx], y_train[amp_tr_idx],
    X_train[amp_va_idx], y_train[amp_va_idx],
    params=best_params_tf_amp,
    seed=12345,
)

train_pred_amp, train_a_amp, train_b_amp, train_b_ratio_amp = predict_transformer_amp(final_tf_amp, X_train)
test_pred_amp, test_a_amp, test_b_amp, test_b_ratio_amp = predict_transformer_amp(final_tf_amp, X_test)

print(f"Train RMSE (reference): {np.sqrt(mean_squared_error(y_train, train_pred_amp)):.3f}")
print(f"b/4.8 train range: [{train_b_ratio_amp.min():.3f}, {train_b_ratio_amp.max():.3f}]")
print(f"b/4.8 test  range: [{test_b_ratio_amp.min():.3f}, {test_b_ratio_amp.max():.3f}]")

test_pred_amp = np.clip(test_pred_amp, 0, None)

sub_amp = pd.DataFrame({
    "sample number": test_df["sample number"],
    "含水率": test_pred_amp,
})
out_sub_amp = "data/sub_transformer_feature_tokens_amp.csv"
sub_amp.to_csv(out_sub_amp, index=False, header=False, encoding="cp932")

dbg_amp = pd.DataFrame({
    "sample number": test_df["sample number"],
    "a": test_a_amp,
    "b": test_b_amp,
    "b_div_4_8": test_b_ratio_amp,
    "pred": test_pred_amp,
})
out_dbg_amp = "data/sub_transformer_feature_tokens_amp_debug.csv"
dbg_amp.to_csv(out_dbg_amp, index=False, header=False, encoding="cp932")

print(f"✅ 保存完了: {out_sub_amp}")
print(f"✅ 保存完了: {out_dbg_amp}")


🎯 AMP版 最終学習 & テスト予測
Train RMSE (reference): 12.481
b/4.8 train range: [0.400, 0.696]
b/4.8 test  range: [0.427, 0.697]
✅ 保存完了: data/sub_transformer_feature_tokens_amp.csv
✅ 保存完了: data/sub_transformer_feature_tokens_amp_debug.csv
